In [34]:
import duckdb
import pandas as pd
con = duckdb.connect("funnel.duckdb")

In [35]:
con.sql("SELECT * FROM accounts LIMIT 5").df()

,account_id,company,segment,industry,country
0,1,"Rodriguez, Figueroa and Sanchez",midmarket,Logistics,USA
1,2,Doyle Ltd,startup,Healthcare,Germany
2,3,"Mcclain, Miller and Henderson",midmarket,Logistics,Germany
3,4,Davis and Sons,midmarket,Manufacturing,UK
4,5,"Guzman, Hoffman and Baldwin",startup,Retail,USA


In [36]:
con.sql("SELECT segment, COUNT(*) AS companies FROM accounts GROUP BY segment ORDER BY companies DESC").df()

,segment,companies
0,startup,361
1,midmarket,184
2,enterprise,55


In [37]:
##-Query 1

con.sql("""
SELECT a.segment,
       COUNT(*) AS leads,
       COUNT(d.deal_id) AS deals,
       COUNT(CASE WHEN d.status = 'won' THEN 1 END) AS won
FROM leads l
JOIN accounts a ON a.account_id = l.account_id
LEFT JOIN deals d ON d.account_id = l.account_id
GROUP BY a.segment
ORDER BY leads DESC
""").df()

,segment,leads,deals,won
0,startup,361,227,102
1,midmarket,184,109,52
2,enterprise,55,44,12


In [38]:
##-Query 2

con.sql("""
SELECT a.segment,
       COUNT(*) AS leads,
       COUNT(d.deal_id) AS deals,
       COUNT(CASE WHEN d.status = 'won' THEN 1 END) AS won,
       ROUND(100.0 * deals / leads, 1) AS pct_leads_to_deal,
       ROUND(100.0 * COUNT(CASE WHEN d.status = 'won' THEN 1 END) / deals, 1) AS pct_deals_won,
FROM leads l
JOIN accounts a ON a.account_id = l.account_id
LEFT JOIN deals d ON d.account_id = l.account_id
GROUP BY a.segment
ORDER BY leads DESC
""").df()

,segment,leads,deals,won,pct_leads_to_deal,pct_deals_won
0,startup,361,227,102,62.9,44.9
1,midmarket,184,109,52,59.2,47.7
2,enterprise,55,44,12,80.0,27.3


In [43]:
##-Query 3

con.sql("""
SELECT a.segment,
         COUNT(*) AS won_customers,
         COUNT(d.churn_date) AS churned,
         ROUND(100.0 * COUNT(d.churn_date) / COUNT(*), 1) AS pct_churned
FROM deals d
JOIN accounts a ON a.account_id = d.account_id
WHERE d.status = 'won'
GROUP BY a.segment
ORDER BY pct_churned DESC
""").df()

,segment,won_customers,churned,pct_churned
0,startup,102,60,58.8
1,midmarket,52,19,36.5
2,enterprise,12,3,25.0


In [48]:
##-Query 4

con.sql("""
SELECT date_trunc('month', u.usage_date) AS month,
       a.segment,
       COUNT(DISTINCT u.account_id) AS active_customers
FROM daily_usage u
JOIN accounts a ON u.account_id = a.account_id
GROUP BY month, a.segment
ORDER BY month, a.segment
""").df()

,month,segment,active_customers
0,2024-04-01,midmarket,1
1,2024-04-01,startup,1
2,2024-05-01,midmarket,3
3,2024-05-01,startup,9
4,2024-06-01,midmarket,4
...,...,...,...
61,2026-02-01,midmarket,33
62,2026-02-01,startup,47
63,2026-03-01,enterprise,11
64,2026-03-01,midmarket,33
